# Build knowledge graph


In [ ]:
%env AWS_PROFILE=kmnlp

In [ ]:
import json  # noqa: I001
import re
from datetime import UTC, datetime
from typing import Any

import s3fs  # pyright: ignore[reportMissingTypeStubs]
from rich import print as rprint  # pyright: ignore[reportUnusedImport] # noqa: F401

from llm_agent.utils.knowledge_graph_construction.graph_construction import build_graph, save_graph
from llm_agent.utils.knowledge_graph_construction.metadata_parsing import (
    Dimension,
    Variable,
    Dataset,
    parse,
)

## Load metadata


### Load metadata for CRW SST dataset


In [ ]:
s3 = s3fs.S3FileSystem()

with s3.open("s3://data-c6c22a2e42294c11b52ee7f0c792c071/crw/ds_metadata.json", "r") as f:  # pyright: ignore[reportUnknownMemberType]
    ds_metadata = json.load(f)

---


## Custom overrides


### Variable names


In [ ]:
variable_field_names = {
    "sst_daily": ["analysed_sst", "sea_ice_fraction"],
    "sst_monthly_mean": ["sea_surface_temperature", "mask"],
    "sst_monthly_min": ["sea_surface_temperature", "mask"],
    "sst_monthly_max": ["sea_surface_temperature", "mask"],
    "sst_annual_mean": ["sea_surface_temperature", "mask"],
    "sst_annual_min": ["sea_surface_temperature", "mask"],
    "sst_annual_max": ["sea_surface_temperature", "mask"],
}

## Parse metadata


In [ ]:
def _get_variable_field_names(
    dataset_name: str,  # noqa: ARG001
    dataset_group: str,
    collection_metadata: dict[str, Any],  # noqa: ARG001
) -> list[str]:
    return variable_field_names[dataset_group]


def _get_metadata_link(
    dataset_name: str,  # noqa: ARG001
    dataset_group: str,
    collection_metadata: dict[str, Any],
) -> str | None:
    if dataset_group == "sst_daily":
        return "https://coralreefwatch.noaa.gov/product/5km/index.php"
    return collection_metadata.get("metadata_link")

In [ ]:
# rprint(
#     parse(
#         "crw_5km",
#         "sst_annual_mean",
#         ds_metadata["crw_5km"]["sst_annual_mean"],
#     )
# )

In [ ]:
datasets: list[Dataset] = []
for ds_name, groups in ds_metadata.items():
    for group_name, group_data in groups.items():
        if group_data["uri"]["parquet"] is None:
            continue
        try:
            dataset = parse(
                ds_name,
                group_name,
                group_data,
                get_variable_field_names_func=_get_variable_field_names,
                get_metadata_link_func=_get_metadata_link,
            )
            datasets.append(dataset)
        except KeyError:
            print("Failed:", ds_name, group_name)  # noqa: T201
            continue


---


## Populate graph


In [ ]:
g = build_graph(
    datasets,
    canonical_dims=True,
    media_type="application/kerchunk-parquet",
)

## Save


In [ ]:
out_path = "../../src/llm_agent/data/dataset-kg-crw-generated.ttl"
save_graph(g, out_path)